In [ ]:
import torch
import torch.nn as nn

"""
- 如果你希望这个张量通过优化器更新，且随模型保存，请用 nn.Parameter。
- 如果你希望这个张量随模型移动设备且随模型保存，但不更新梯度，请用 register_buffer。
- 如果是临时中间变量，请用普通的 Tensor。
"""
class DemoModule(nn.Module):
    def __init__(self):
        super().__init__()
        # 1. Parameter: 优化器会更新它
        self.weights = nn.Parameter(torch.randn(2, 2))
        
        # 2. Buffer: 不更新梯度，但会随模型保存（如推理时的起始位置）
        self.register_buffer("mask", torch.ones(2, 2))
        
        # 3. 普通属性: 只是一个临时的 Tensor，模型保存时会丢失
        self.temp_data = torch.zeros(2, 2)

model = DemoModule()

print("--- 权重管理检查 ---")
print(f"1. parameters() 中包含的对象: {[n for n, p in model.named_parameters()]}")
print(f"2. buffers() 中包含的对象: {[n for n, b in model.named_buffers()]}")
print(f"3. state_dict (模型保存内容) 中包含: {model.state_dict().keys()}")

# 如果你定义了 RoPE 的频率 theta，且不希望它被更新，
# 但希望 model.to('cuda') 时它也一起过去，请务必使用 register_buffer！

--- 权重管理检查 ---
1. parameters() 中包含的对象: ['weights']
2. buffers() 中包含的对象: ['mask']
3. state_dict (模型保存内容) 中包含: odict_keys(['weights', 'mask'])


In [6]:
for n, p in model.named_parameters():
    print(f"Parameter: {n}, requires_grad={p.requires_grad}")
    print(p)

Parameter: weights, requires_grad=True
Parameter containing:
tensor([[-1.0837, -0.6418],
        [ 0.8675,  1.7943]], requires_grad=True)


In [ ]:
import torch
import torch.nn as nn
import os

# 1. 定义一个简单的模型
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 定义一个 nn.Parameter
        self.embedding_table = nn.Parameter(torch.randn(10, 8))
        self.register_buffer("counter", torch.tensor(0)) 
        self.temp_data = torch.zeros(2, 2)

    def forward(self, x):
        return self.embedding_table[x]

# --- 第一阶段：初始化并保存 ---
model = SimpleModel()
print("1. 初始参数前两行:\n", model.embedding_table[:2])


# 准备保存的 Checkpoint 字典
# 工业界标准：不仅保存模型权重，还要保存优化器状态和步数
checkpoint = {
    "model_state_dict": model.state_dict(),
    "iter": 100,
    "config": {"vocab_size": 10, "d_model": 8}
}

ckpt_path = "checkpoint.pt"
torch.save(checkpoint, ckpt_path)
print(f"\n✅ Checkpoint 已保存至 {ckpt_path}")

# --- 第二阶段：加载并读取 nn.Parameter ---
# 创建一个新的模型实例（模拟重新启动程序）
new_model = SimpleModel()
print("\n2. 新实例加载前的参数（随机初始化）:\n", new_model.embedding_table[:2])

# 加载磁盘文件
loaded_ckpt = torch.load(ckpt_path)

# 将权重加载进模型
new_model.load_state_dict(loaded_ckpt["model_state_dict"])
print("\n3. 加载后的参数:\n", new_model.embedding_table[:2])

for name, param in new_model.named_parameters():
    print(f"Parameter: {name}, requires_grad={param.requires_grad}")
    

# 方式 C：直接从加载的字典里看（不经过模型）
raw_tensor = loaded_ckpt["model_state_dict"]["embedding_table"]
print(f"直接从字典读取的张量形状: {raw_tensor.shape}")

# 清理文件
if os.path.exists(ckpt_path):
    os.remove(ckpt_path)

In [2]:
# 创建一个权重矩阵并初始化
w = nn.Parameter(torch.empty(1000, 100))

# 执行截断正态分布初始化
# mean=0, std=0.02 (LLM 常用值), 截断在 [-2, 2] 倍标准差
nn.init.trunc_normal_(w, mean=0.0, std=0.02, a=-0.04, b=0.04)

print("--- 初始化统计 ---")
print(f"最大值: {w.max().item():.4f}")
print(f"最小值: {w.min().item():.4f}")
print(f"标准差: {w.std().item():.4f}")

# 观察：你会发现没有任何一个值会超出 [a, b] 范围

--- 初始化统计 ---
最大值: 0.0400
最小值: -0.0400
标准差: 0.0176
